In [ ]:
import json
import math
import matplotlib.pyplot as plt
from collections import defaultdict

# =========================
# Load metrics
# =========================
loss_dict = defaultdict(list)  # iteration -> list of losses
ap_dict = {}  # iteration -> ap_value

json_file = "/mnt/HDD4/anlt/selected_topics/Selected_Topics/HW3/checkpoints/mpvit/mask_rcnn_mpvit_tiny_ms_3x_r101_fpn_200/metrics.json"

with open(json_file, "r") as f:
    for i, line in enumerate(f):
        try:
            entry = json.loads(line)
            it = entry.get("iteration", None)

            if it is not None and "total_loss" in entry:
                loss_dict[it].append(entry["total_loss"])

            if it is not None and "segm/AP50" in entry:
                ap_dict[it] = entry["segm/AP50"]  # assume only one per iteration

        except json.JSONDecodeError:
            continue

# Average losses per iteration
iterations_loss = sorted(loss_dict.keys())
losses = [sum(loss_dict[it]) / len(loss_dict[it]) for it in iterations_loss]

# AP values
iterations_ap = sorted(ap_dict.keys())
ap_values = [ap_dict[it] for it in iterations_ap]


# =========================
# Create 1x2 subplot
# =========================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# =========================
# Training Loss Curve
# =========================
axes[0].plot(
    iterations_loss,
    losses,
    linewidth=1,
    label="Training Loss (averaged)"
)

axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Total Loss")
axes[0].set_title("Training Loss Curve")
axes[0].legend()

if losses:
    min_loss = min(losses)
    min_loss_it = iterations_loss[losses.index(min_loss)]

    axes[0].scatter(min_loss_it, min_loss)

    axes[0].annotate(
        f"Min Loss = {min_loss:.3f}",
        xy=(min_loss_it, min_loss),
        xytext=(10, 10),
        textcoords="offset points",
    )


# =========================
# Validation AP Curve
# =========================
axes[1].plot(
    iterations_ap,
    ap_values,
    linewidth=1.5,
    marker="o",
    markersize=4,
    label="Validation segm/AP"
)

axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("segm/AP50")
axes[1].set_title("Validation Segmentation AP")
axes[1].legend()

if ap_values:
    max_ap = max(ap_values)
    max_ap_it = iterations_ap[ap_values.index(max_ap)]

    axes[1].scatter(max_ap_it, max_ap)

    axes[1].annotate(
        f"Best AP = {max_ap:.2f}",
        xy=(max_ap_it, max_ap),
        xytext=(10, 10),
        textcoords="offset points",
    )

plt.tight_layout()
plt.show()

In [ ]:
import os
import json
import torch

from detectron2.config import get_cfg
from detectron2.modeling import build_model
from detectron2.checkpoint import DetectionCheckpointer

from mpvit import add_mpvit_config

# Root directory containing experiment folders
root_dir = "checkpoints/mpvit"

results = []

for folder_name in sorted(os.listdir(root_dir)):
    folder_path = os.path.join(root_dir, folder_name)

    if not os.path.isdir(folder_path):
        continue

    metrics_file = os.path.join(folder_path, "metrics.json")
    config_file = os.path.join(folder_path, "config.yaml")
    model_file = os.path.join(folder_path, "model_best.pth")

    if not os.path.exists(metrics_file):
        print(f"[Skip] No metrics.json in {folder_name}")
        continue

    # =========================
    # Read max AP50
    # =========================
    max_ap50 = None

    with open(metrics_file, "r") as f:
        for line in f:
            try:
                entry = json.loads(line)

                if "segm/AP50" in entry:
                    ap50 = entry["segm/AP50"]

                    if max_ap50 is None or ap50 > max_ap50:
                        max_ap50 = ap50

            except json.JSONDecodeError:
                continue

    # =========================
    # Build model
    # =========================
    cfg = get_cfg()
    add_mpvit_config(cfg)
    cfg.merge_from_file(config_file)
    cfg.MODEL.DEVICE = "cpu"

    model = build_model(cfg)

    # Load checkpoint
    if os.path.exists(model_file):
        DetectionCheckpointer(model).load(model_file)

    # =========================
    # Count params
    # =========================
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    results.append({
        "model": folder_name,
        "ap50": max_ap50,
        "params_million": total_params / 1e6,
        "trainable_million": trainable_params / 1e6,
    })


# =========================
# Print
# =========================
print("\n===== Results =====")

for r in results:
    print(
        f"{r['model']:<40} "
        f"AP50: {r['ap50']:<8.2f} "
        f"Params: {r['params_million']:.2f}M "
        f"Trainable: {r['trainable_million']:.2f}M"
    )